In [0]:
BRONZE_PATH = "/Volumes/workspace/default/logs_volume/bronze"
SILVER_PATH = "/Volumes/workspace/default/logs_volume/silver"

In [0]:
# Read from the bronze layer, previous layer
bronze_df = spark.read.format("delta").load(BRONZE_PATH)
print(f"Rows loaded from Bronze: {bronze_df.count():,}")

In [0]:
import re
# Each group in the pattern captures one field
LOG_PATTERN = r'^(\S+) \S+ \S+ \[([^\]]+)\] "(\S+) (\S+) (\S+)" (\d{3}) (\S+)'

sample = '199.72.81.55 - - [01/Jul/1995:00:00:01 -0400] "GET /history/apollo/ HTTP/1.0" 200 6245'
print(re.match(LOG_PATTERN, sample).groups())

In [0]:
from pyspark.sql.functions import regexp_extract, col, when

parsed_df = bronze_df.select(
    regexp_extract("value", LOG_PATTERN, 1).alias("host"),
    regexp_extract("value", LOG_PATTERN, 2).alias("timestamp_raw"),
    regexp_extract("value", LOG_PATTERN, 3).alias("method"),
    regexp_extract("value", LOG_PATTERN, 4).alias("endpoint"),
    regexp_extract("value", LOG_PATTERN, 5).alias("protocol"),
    regexp_extract("value", LOG_PATTERN, 6).alias("status_code_raw"),
    regexp_extract("value", LOG_PATTERN, 7).alias("bytes_raw")
)

parsed_df.show(5, truncate=False)

In [0]:
from pyspark.sql.functions import col, when, try_to_timestamp, lit

silver_df = parsed_df.select(
    col("host"),

    try_to_timestamp(
        when(col("timestamp_raw") == "", lit(None))
        .otherwise(col("timestamp_raw")),
        lit("dd/MMM/yyyy:HH:mm:ss Z")
    ).alias("timestamp"),

    col("method"),
    col("endpoint"),
    col("protocol"),

    when(col("status_code_raw") == "", lit(None))
    .otherwise(col("status_code_raw"))
    .cast("int")
    .alias("status"),

    when((col("bytes_raw") == "-") | (col("bytes_raw") == ""), 0)
    .otherwise(col("bytes_raw"))
    .cast("int")
    .alias("bytes")
)

silver_df.printSchema()

silver_df.show(5, truncate=False)

In [0]:
# Check how many rows failed to parse
bad_rows = silver_df.filter(col("host") == "")
print(f"Malformed rows: {bad_rows.count():,}")

# Keep only valid rows
clean_df = silver_df.filter(col("host") != "")
print(f"Clean rows: {clean_df.count():,}")

In [0]:
"""
  Why partitionBy("status_code")? When we later query "show me all 404 errors", Spark only reads the status_code=404 partition folder instead of scanning the entire table. This is a key performance optimization.
"""
(clean_df
  .write
  .format("delta")
  .mode("overwrite")
  .partitionBy("status")   # partitioning makes future queries faster
  .save(SILVER_PATH))



print("Silver layer written successfully.")

In [0]:
# Verification
silver_df = spark.read.format("delta").load(SILVER_PATH)

print(f"Total clean rows: {silver_df.count():,}")
print("\nStatus code distribution:")
silver_df.groupBy("status").count().orderBy("count", ascending=False).show()